# Pixel opportunity for a no-candidate Aqua observation

This notebook validates the Aqua and NOAA-20 geolocation downloads, checks whether the NOAA-20 same-acquisition reprocessed geolocation agrees with all fire-pixel coordinates embedded in the NOAA-20 L2 product at its stored `FP_line` / `FP_sample` indices, and then samples the paired native fire masks at both the Aqua and NOAA-20 observation coordinates. The NOAA-20 L2 metadata named a prior geolocation processing timestamp that CMR no longer returns; the downloaded reprocessing is used only after the coordinate consistency audit.

The Aqua–NOAA-20 overpasses are 33 minutes apart. This single case does not estimate accuracy or prove event identity.

In [1]:
from pathlib import Path
import sys,json,hashlib
import numpy as np
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/reference').is_dir())
sys.path.insert(0,str(ROOT))
from src.firewatch.coverage_validation import paired_modis_point,paired_viirs_point,sha256_file,_distance_km
geo_plan_path=ROOT/'data/reference/unlinked_opportunity_geolocation_plan_2025.csv'
geo_dir=ROOT/'data/reference/granules/unlinked_opportunity_geolocation_pilot'
l2_dir=ROOT/'data/reference/granules/validation_2025'
n20_l2=ROOT/'data/reference/granules/unlinked_opportunity_pilot/VJ114IMG.A2025261.1330.002.2025261205555.nc'
aqua_l2=ROOT/'data/reference/granules/validation_2025/MYD14.A2025261.1405.061.2025262112616.hdf'
plan=pd.read_csv(geo_plan_path)
manifest=json.loads((geo_dir/'download_manifest.json').read_text(encoding='utf-8'))
assert manifest['plan_sha256']==hashlib.sha256(geo_plan_path.read_bytes()).hexdigest()
for rec in manifest['downloads']:
    p=geo_dir/rec['filename']; assert p.stat().st_size==rec['bytes']; assert sha256_file(p)==rec['sha256']
l2manifest=json.loads((n20_l2.parent/'download_manifest.json').read_text(encoding='utf-8'))
n20rec=next(x for x in l2manifest['downloads'] if x['filename']==n20_l2.name)
assert n20_l2.stat().st_size==n20rec['bytes'] and sha256_file(n20_l2)==n20rec['sha256']
sample=pd.read_csv(ROOT/'data/processed/source_pixel_validation_2025_stratified.csv')
aqua=sample[sample.observation_id.eq(plan.iloc[0].source_l2_filename) ] if False else sample[(sample.satellite=='Aqua')&(sample.quarter=='Q3')&(sample.candidate_status==False)].iloc[0]
# Tie point coordinates to the selected source observation from the plan.
refrow=pd.read_csv(ROOT/'data/reference/unlinked_opportunity_l2_plan_2025.csv').iloc[0]
aqua=sample[sample.observation_id.eq(refrow.source_observation_id)].iloc[0]
aqua_lat,aqua_lon=float(aqua.csv_latitude),float(aqua.csv_longitude)
display(pd.DataFrame([{'file':r['filename'],'bytes':r['bytes'],'sha256':r['sha256']} for r in manifest['downloads']]))
print('Aqua target:',aqua_lat,aqua_lon,'at',aqua.observation_time_utc)

,file,bytes,sha256
0,MYD03.A2025261.1405.061.2025262150419.hdf,33260201,84b7948a73ba2a2626bc60e915db1f318a21df0b40c6e0...
1,VJ103IMG.A2025261.1330.021.2025267192836.nc,191223131,5061c25308ac6da716bbe6e48ff911a835355608af4ccf...


Aqua target: 4.0848 9.3146 at 2025-09-18T14:06:00+00:00


In [2]:
import h5py
n20_geo=geo_dir/plan.loc[plan.source_l2_filename.eq(n20_l2.name),'geolocation_granule_id'].iloc[0]
with h5py.File(n20_l2,'r') as fire,h5py.File(n20_geo,'r') as geo:
    line=fire['FP_line'][:].astype(int)
    sample_index=fire['FP_sample'][:].astype(int)
    fire_lat=fire['FP_latitude'][:].astype(float)
    fire_lon=fire['FP_longitude'][:].astype(float)
    lat_ds=geo['geolocation_data/latitude']; lon_ds=geo['geolocation_data/longitude']
    assert lat_ds.shape==fire['fire mask'].shape
    indexed_lat=np.array([lat_ds[r,c] for r,c in zip(line,sample_index)],dtype=float)
    indexed_lon=np.array([lon_ds[r,c] for r,c in zip(line,sample_index)],dtype=float)
    residual_km=_distance_km(indexed_lat,indexed_lon,fire_lat,fire_lon)
    qflags=geo['geolocation_data/quality_flag']
    fire_qflags=np.array([qflags[r,c] for r,c in zip(line,sample_index)],dtype=int)
audit={'active_fire_records':len(line),'median_residual_m':float(np.median(residual_km)*1000),'p95_residual_m':float(np.percentile(residual_km,95)*1000),'max_residual_m':float(np.max(residual_km)*1000),'records_under_0_1m':int((residual_km<0.0001).sum()),'records_under_1m':int((residual_km<0.001).sum()),'quality_flags_at_fire_pixels':sorted(set(fire_qflags.tolist()))}
display(pd.DataFrame([audit]))
print('Original L2-named NOAA-20 geolocation file is absent from current CMR; this is a reprocessed same-acquisition candidate. Pixel-index agreement above is the acceptance evidence for this case.')

,active_fire_records,median_residual_m,p95_residual_m,max_residual_m,records_under_0_1m,records_under_1m,quality_flags_at_fire_pixels
0,16,0.0,0.0,0.0,16,16,[0]


Original L2-named NOAA-20 geolocation file is absent from current CMR; this is a reprocessed same-acquisition candidate. Pixel-index agreement above is the acceptance evidence for this case.


In [3]:
aqua_geo=geo_dir/plan.loc[plan.source_l2_filename.eq(aqua_l2.name),'geolocation_granule_id'].iloc[0]
aqua_result=paired_modis_point(aqua_geo,aqua_l2,aqua_lat,aqua_lon)
n20_result=paired_viirs_point(n20_geo,n20_l2,aqua_lat,aqua_lon)
# Also verify the NOAA-20's own embedded detection coordinate maps to its fire mask.
own=sample[sample.satellite.eq('N20') & sample.quarter.eq('Q3') & sample.candidate_status & sample.candidate_status_label.eq('has_candidate_link')].iloc[0]
# This Q3 linked N20 point is in July, so instead select the embedded NOAA-20 fire pixel nearest the September Aqua target.
with h5py.File(n20_l2,'r') as fire:
    fp_lat=fire['FP_latitude'][:]; fp_lon=fire['FP_longitude'][:]
    k=int(np.argmin(_distance_km(fp_lat,fp_lon,aqua_lat,aqua_lon)))
    closest_fire={'latitude':float(fp_lat[k]),'longitude':float(fp_lon[k]),'distance_from_aqua_km':float(_distance_km(np.array([fp_lat[k]]),np.array([fp_lon[k]]),aqua_lat,aqua_lon)[0]),'line':int(fire['FP_line'][k]),'sample':int(fire['FP_sample'][k])}
rows=[{'target':'Aqua no-candidate coordinate','swath':'Aqua MODIS','overpass_utc':aqua.observation_time_utc,**aqua_result},{'target':'Aqua no-candidate coordinate','swath':'NOAA-20 VIIRS','overpass_utc':refrow.granule_start_utc,**n20_result}]
coverage=pd.DataFrame(rows)
coverage.to_csv(ROOT/'data/processed/unlinked_opportunity_pixel_coverage_2025.csv',index=False)
display(coverage[['target','swath','distance_km','fire_mask_class','fire_mask_label','gflags','quality_flag','quality_flag_usable']])
display(pd.DataFrame([closest_fire]))
print('Saved data/processed/unlinked_opportunity_pixel_coverage_2025.csv')

,target,swath,distance_km,fire_mask_class,fire_mask_label,gflags,quality_flag,quality_flag_usable
0,Aqua no-candidate coordinate,Aqua MODIS,0.003753,8,fire_nominal_confidence,0.0,NaN,NaN
1,Aqua no-candidate coordinate,NOAA-20 VIIRS,0.239900,1,bowtie,NaN,0.0,True


,latitude,longitude,distance_from_aqua_km,line,sample
0,4.515252,7.758847,179.021866,212,746


Saved data/processed/unlinked_opportunity_pixel_coverage_2025.csv


## Interpretation

The NOAA-20 reprocessed geolocation is compared at all 16 embedded fire-pixel indices. Sub-meter residuals support using this reprocessing for the sampled NOAA-20 fire pixels and nearby case-study inspection, although they do not prove geolocation equality at every swath location. The Aqua coordinate is then mapped to the nearest valid center in each swath and the corresponding source mask classes are read. The NOAA-20 target time is the Level-2 granule interval around 13:33 UTC; Aqua is around 14:06 UTC. Any clear-land class at Aqua's later coordinate is evidence of the earlier product's classification there, not proof of a NOAA-20 detection miss. This remains one selected point and requires broader sampling for performance claims.